# Reading and interpreting the `.jld2` datasets

The generation scripts in [`scripts/`](../scripts) write their output as JLD2
files. This notebook collects the functions needed to load those files back and
make sense of their contents. Four artifact shapes exist:

| file | written by | one entry is |
| --- | --- | --- |
| `pncp_NxM.jld2` | `gen_pncp.jl` | a PnCP form — Choi matrix of a positive, non-CP map |
| `ppt_entangled_NxM.jld2` | `gen_ppt.jl` | an entangled PPT state ρ |
| `detection_NxM.jld2` | `compare_detection.jl` | a named tuple: a state plus every criterion's score |
| `result_<i>_<j>.jld2` | `test_ppt2.jl` | one flagged PPT² composite and its witnesses |

The first three are **batched**: entries live under `batch_<id>` keys and
dataset metadata under a `meta` group. The result files are flat, one composite
per file.

In [ ]:
include("common.jl")   # ppt2 (ampliation, is_ppt, …), JLD2, Ket, Mosek, detection helpers

## Loading the batched datasets

`load_dataset` concatenates every `batch_<id>` group **in id order** into one
vector (matrices for the generators, named tuples for `compare_detection`).
Order matters: the indices stored in result/detection files (`dot_idx`,
`amp_idx`, the `i`/`j` of a composite) point into exactly this ordering.
`load_meta` returns the `meta` group as a Dict — dataset values (`dim_A`,
`dim_B`, `tol`, `level`) and the per-batch `attempted`/`accepted` counters — and
is empty for older files that carry no metadata.

In [ ]:
batch_id(key) = parse(Int, split(key, "_")[2])

"Concatenate every `batch_<id>` group, in id order. Skips the `meta` group."
function load_dataset(path)
    jldopen(path, "r") do f
        ks = sort([k for k in keys(f) if startswith(k, "batch_")]; by = batch_id)
        isempty(ks) ? [] : reduce(vcat, f[k] for k in ks)
    end
end

"All `meta` entries as a Dict (empty when the file carries no metadata)."
function load_meta(path)
    jldopen(path, "r") do f
        haskey(f, "meta") || return Dict{String,Any}()
        g = f["meta"]
        Dict{String,Any}(k => g[k] for k in keys(g))
    end
end

"One-line summary of a batched dataset: size, recorded dims/tol, acceptance rate."
function describe(path)
    meta  = load_meta(path)
    items = load_dataset(path)
    att = sum(v for (k, v) in meta if endswith(k, "_attempted"); init = 0)
    acc = sum(v for (k, v) in meta if endswith(k, "_accepted");  init = 0)
    sz  = isempty(items) ? missing :
          (items[1] isa AbstractMatrix ? size(items[1]) : size(items[1].state))
    (file = basename(path), items = length(items),
     dims = haskey(meta, "dim_A") ? (meta["dim_A"], meta["dim_B"]) : missing,
     matrix_size = sz, tol = get(meta, "tol", missing),
     attempted = att, accepted = acc,
     accept_pct = att > 0 ? round(100acc / att, digits = 2) : missing)
end

In [ ]:
describe.(["../pncp_4x4.jld2", "../ppt_entangled_4x4.jld2"])

## PnCP forms — `pncp_NxM.jld2`

Each entry is the Choi matrix `M` (size `nm × nm`) of a positive map that is
**not** completely positive — the matrix counterpart of the non-SOS biquadratic
certificate built in [`pncp.jl`](../src/pncp.jl). Two checks interpret it:

- **not completely positive** ⇔ `M` is not PSD, i.e. `eigmin(M) < 0`
  (`is_completely_positive` below);
- **positive map** ⇔ `⟨x⊗y| M |x⊗y⟩ ≥ 0` for every product vector, verified by
  the see-saw SDP `min_xy_form` from `common.jl`.

In [ ]:
is_completely_positive(form; tol = 1e-8) = eigmin(Hermitian(Matrix(form))) ≥ -tol

"Verify one form is a positive (but not completely positive) map."
function interpret_form(form, n, m)
    min_prod, _, _ = min_xy_form(form, n, m)   # min ⟨x⊗y| form |x⊗y⟩ via see-saw SDP
    (matrix_size = size(form),
     min_eig = eigmin(Hermitian(Matrix(form))),
     completely_positive = is_completely_positive(form),
     min_product_value = min_prod,
     positive_map = min_prod ≥ -1e-6)
end

forms = load_dataset("../pncp_4x4.jld2")
println(length(forms), " forms; all non-CP? ", all(f -> !is_completely_positive(f), forms))
interpret_form(forms[1], 4, 4)

## Entangled PPT states — `ppt_entangled_NxM.jld2`

Each entry is a density matrix ρ that is PSD, has positive partial transpose,
yet is entangled (its level-2 DPS robustness exceeded `tol` at generation time).
`is_ppt` (from `ppt2`) checks the partial transpose; the three `detect_*`
helpers from `common.jl` re-certify the entanglement: `:trace` and `:ampliation`
use the PnCP form library, `:dps` re-runs the SDP. On these bound-entangled
states the linear PnCP witnesses often miss what DPS still catches — exactly the
comparison the detection dataset below quantifies.

In [ ]:
states = load_dataset("../ppt_entangled_4x4.jld2")
ρ = states[1]
(psd        = eigmin(Hermitian(ρ)) ≥ -1e-8,
 ppt        = is_ppt(ρ, 4, 4),
 trace      = detect_trace(ρ, forms),
 ampliation = detect_ampliation(ρ, forms, 4, 4),
 dps        = detect_dps(ρ, 4, 4).detected)

## Detection comparison — `detection_NxM.jld2`

`compare_detection.jl` records, per kept state, the raw score of **every**
criterion so the methods can be compared after the fact:

| field | criterion | entangled when |
| --- | --- | --- |
| `robustness` | DPS (level `level`) | `> tol` |
| `min_dot` | min `tr(form·ρ)` over forms | `< -tol` |
| `min_amp` | min `λ_min((I⊗form)(ρ))` over forms | `< -tol` |

(`dot_idx`/`amp_idx` are the forms achieving those minima.) `summarize_detection`
rebuilds the per-criterion efficacy table — including DPS-only and PnCP-only
counts — straight from the file, using its recorded `tol`/`level`.

In [ ]:
function summarize_detection(path)
    meta  = load_meta(path)
    tol   = get(meta, "tol", 1e-8)
    level = get(meta, "level", 2)
    R = load_dataset(path)
    N = length(R)
    N == 0 && return println("empty dataset")

    dps   = [r.robustness > tol for r in R]
    trace = [r.min_dot < -tol for r in R]
    ampl  = [r.min_amp < -tol for r in R]
    pncp  = trace .| ampl

    pct(c)        = round(100c / N, digits = 2)
    line(lbl, mk) = println("  ", rpad(lbl, 28), rpad(count(mk), 7), "(", pct(count(mk)), "%)")

    println("Detection efficacy over ", N, " states (each detected by ≥1 criterion):")
    line("DPS (level $level)", dps)
    line("PnCP trace witness", trace)
    line("PnCP ampliation", ampl)
    line("PnCP (trace OR ampliation)", pncp)
    println("  ", "-"^36)
    line("DPS only (PnCP missed)", dps .& .!pncp)
    line("PnCP only (DPS missed)", pncp .& .!dps)
    line("DPS and PnCP", dps .& pncp)
end

det = "../detection_4x4.jld2"
isfile(det) ? summarize_detection(det) :
    println("No $det yet — create one with `scripts/compare_detection.jl`.")

## PPT² test results — `result_<i>_<j>.jld2`

When `test_ppt2.jl` flags a composite τ = (I⊗Φⱼ)(ρᵢ) as possibly entangled it
writes one file per flagged pair. Unlike the datasets these are flat:

| key | meaning |
| --- | --- |
| `i`, `j` | indices of the two states (into the `load_dataset` ordering) |
| `state` | the composite τ |
| `witness` | DPS entanglement witness |
| `dot_idx`, `dot_mat` | form minimising `tr(form·τ)`, and that form |
| `amp_idx`, `amp_mat` | form minimising `λ_min((I⊗form)(τ))`, and that form |

`load_result` reads the file; `inspect_result` recomputes each criterion on the
composite to confirm the flag and reports whether τ is still PPT (a genuine
counterexample would be PPT **and** entangled).

In [ ]:
"Read one `result_<i>_<j>.jld2` into a named tuple."
load_result(path) = jldopen(path, "r") do f
    (i = f["i"], j = f["j"], state = f["state"], witness = f["witness"],
     dot_idx = f["dot_idx"], dot_mat = f["dot_mat"],
     amp_idx = f["amp_idx"], amp_mat = f["amp_mat"])
end

"Re-verify a flagged composite: is it still PPT, and which criteria detect it?"
function inspect_result(path; n = 4, m = 4, tol = 1e-8)
    r = load_result(path)
    τ = Hermitian(Matrix(r.state))
    (i = r.i, j = r.j,
     ppt = is_ppt(r.state, n, m),
     min_eig = eigmin(τ),
     trace_score = tr(r.dot_mat * r.state),                              # < -tol ⇒ entangled
     ampliation_score = eigmin(Hermitian(ampliation(r.amp_mat, r.state, n, m))),
     dps_robustness = detect_dps(r.state, n, m).value,                   # > tol ⇒ entangled
     witness_expectation = tr(r.witness * r.state))
end

results = sort(filter(f -> startswith(f, "result_") && endswith(f, ".jld2"), readdir("..")))
isempty(results) ? println("No result_*.jld2 files in ../ — run `scripts/test_ppt2.jl`.") :
    inspect_result(joinpath("..", results[1]))